# 508 — Kong MCP Gateway: App Client Integration

**Phase 508** wires `KongMCPToolClient` into the full application stack so that
all MCP tool calls from every agent route through Kong when
`KONG_MCP_GATEWAY_ENABLED=true`.

Phase 507 was transport plumbing — smoke tests only, no app changes.
Phase 508 is the actual switchover.

## What this notebook covers

| Part | Topic |
|------|---------|
| 1 | What changed in Phase 508 |
| 2 | Three-mode MCP architecture |
| 3 | Environment inspection |
| 4 | `KongMCPToolClient` — construction and interface |
| 5 | Factory wiring — `create_app_components(mcp_mode=...)` |
| 6 | Live tests (gated by `ENABLE_LIVE_KONG_NOTEBOOK_TESTS`) |
| 7 | Fail-closed behavior |
| 8 | Regression: local and remote MCP still work |
| 9 | UI sidebar: Kong as a third option |
| 10 | Rollback |

## Prerequisites

- Phase 507 Kong MCP Gateway is provisioned in Konnect (service + route + key-auth).
- `.env` contains `KONG_PROXY_URL`, `KONG_MCP_GATEWAY_API_KEY`, and
  `KONG_MCP_GATEWAY_ENABLED=true`.
- Set `ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true` to run cells that make real HTTP calls.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import os
import json
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

import nest_asyncio
nest_asyncio.apply()  # allow asyncio.run() inside Jupyter's running event loop

print("Environment loaded.")

Environment loaded.


---

## Part 1 — What changed in Phase 508

Phase 507 added the Kong `/mcp` route and proved that the MCP Streamable HTTP
transport works through Kong via raw SDK calls and `requests`.

Phase 508 makes that path usable by the whole application:

| Item | Phase 507 | Phase 508 |
|---|---|---|
| Kong MCP route in Konnect | ✓ provisioned | ✓ unchanged |
| Raw SDK smoke test via Kong | ✓ | ✓ unchanged |
| `KongMCPToolClient` class | ✗ | ✓ new |
| Factory `mcp_mode="kong"` branch | ✗ | ✓ new |
| All agents use Kong when enabled | ✗ | ✓ automatic |
| Streamlit sidebar: Kong option | ✗ | ✓ new |
| Fail-closed (no silent fallback) | N/A | ✓ enforced |

### Feature flag

A single flag controls everything:

```dotenv
KONG_MCP_GATEWAY_ENABLED=true
```

When `false` (the default), the app behaves exactly as it did before Phase 508.

---

## Part 2 — Three-mode MCP architecture

After Phase 508 the app supports three MCP backends, selectable at startup:

```
mcp_mode
  ├── "local"   MCPToolClient        in-process module calls
  │                                   (always available, no network)
  ├── "remote"  RemoteMCPToolClient   direct HTTP → Railway MCP server
  │                                   (requires REMOTE_MCP_URL)
  └── "kong"    KongMCPToolClient     HTTP → Kong proxy → Railway MCP server
                                      (requires KONG_MCP_GATEWAY_ENABLED=true)
```

The selection point is the single factory function:

```python
components = create_app_components(mcp_mode="kong")
```

The `mcp_client` inside `components` is then passed to all three agents
(`GraphAgent`, `RiskAgent`, `TraceAgent`) and the `Orchestrator`.
No per-agent logic was added.

### Kong path breakdown

```
Agent.run()
  └── KongMCPToolClient.call_tool(tool_name, args)
        └── streamablehttp_client(KONG_PROXY_URL + KONG_MCP_GATEWAY_ROUTE_PATH,
                                  headers={"X-Kong-API-Key": KONG_MCP_GATEWAY_API_KEY,
                                           "Accept": "text/event-stream"})
              └── Kong Serverless (key-auth validates, forwards to upstream)
                    └── Railway FastMCP server /mcp
```

---

## Part 3 — Environment inspection

In [3]:
from src.config import (
    get_kong_mcp_gateway_settings,
    get_remote_mcp_url,
)

mcp_gw = get_kong_mcp_gateway_settings()
direct_mcp_url = get_remote_mcp_url()

print("=== Kong MCP Gateway settings ===")
for k, v in mcp_gw.masked().items():
    print(f"  {k}: {v}")

print()
print(f"Direct remote MCP URL : {direct_mcp_url or '(not set)'}")
print()

live = os.getenv("ENABLE_LIVE_KONG_NOTEBOOK_TESTS", "").strip().lower() == "true"
print(f"Live tests enabled    : {live}")
if not live:
    print("  → Set ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true to run HTTP cells.")

=== Kong MCP Gateway settings ===
  enabled: True
  proxy_url: https://kong-948ef205bdausa0ym.kongcloud.dev
  route_path: /mcp
  api_key: KsxhEYlr…***
  upstream_url: https://entity-risk-ai-production.up.railway.app
  gateway_url: https://kong-948ef205bdausa0ym.kongcloud.dev/mcp

Direct remote MCP URL : https://entity-risk-ai-production.up.railway.app/mcp

Live tests enabled    : True


---

## Part 4 — `KongMCPToolClient` — construction and interface

`KongMCPToolClient` lives in `src/clients/kong_mcp_tool_client.py`.
It mirrors the public interface of `RemoteMCPToolClient` so agents need
no changes:

| Method | Behaviour |
|---|---|
| `call_tool(tool_name, arguments)` | Opens Kong session, calls tool, returns `ToolResult`. Raises `RuntimeError` on any failure (fail-closed). |
| `list_tools()` | Returns the sorted list of 15 known tool names (static, no network). |
| `close()` | No-op — stateless, no persistent connections. |

### Construction

The client reads its endpoint and key from `KongMCPGatewaySettings` (already
defined in `src/config.py` since Phase 507).  No new config was added.

In [4]:
from src.clients.kong_mcp_tool_client import KongMCPToolClient
from src.clients.remote_mcp_tool_client import RemoteMCPToolClient
from src.clients.mcp_tool_client import MCPToolClient

print("=== KongMCPToolClient interface ===")
print(f"  call_tool : {KongMCPToolClient.call_tool}")
print(f"  list_tools: {KongMCPToolClient.list_tools}")
print(f"  close     : {KongMCPToolClient.close}")
print()

# Verify the three clients share the same public method signatures
for cls in (MCPToolClient, RemoteMCPToolClient, KongMCPToolClient):
    has_call = callable(getattr(cls, "call_tool", None))
    has_list = callable(getattr(cls, "list_tools", None))
    has_close = callable(getattr(cls, "close", None))
    status = "✓" if all([has_call, has_list, has_close]) else "✗"
    print(f"  {status} {cls.__name__}: call_tool={has_call}, list_tools={has_list}, close={has_close}")

=== KongMCPToolClient interface ===
  call_tool : <function KongMCPToolClient.call_tool at 0x11820fce0>
  list_tools: <function KongMCPToolClient.list_tools at 0x11820fd80>
  close     : <function KongMCPToolClient.close at 0x11820fe20>

  ✓ MCPToolClient: call_tool=True, list_tools=True, close=True
  ✓ RemoteMCPToolClient: call_tool=True, list_tools=True, close=True
  ✓ KongMCPToolClient: call_tool=True, list_tools=True, close=True


In [5]:
# Construct the client (no network call at construction time — just logs endpoint)
# This is safe to run regardless of ENABLE_LIVE_KONG_NOTEBOOK_TESTS

if not mcp_gw.enabled:
    print("KONG_MCP_GATEWAY_ENABLED is false — constructing with enabled=False settings.")
    print("The client would be unusable for live calls but the class loads fine.")

client = KongMCPToolClient(mcp_gw)

print(f"Endpoint : {client._endpoint}")
print(f"Headers  : X-Kong-API-Key=***, Accept=text/event-stream")
print()
print(f"list_tools() (static, no network):")
for name in client.list_tools():
    print(f"  - {name}")

[04/05/26 21:14:31] INFO     Using Kong MCP Gateway                                      ]8;id=455629;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py\kong_mcp_tool_client.py]8;;\:]8;id=554939;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py#62\62]8;;\

                    INFO     Endpoint: https://kong-948ef205bdausa0ym.kongcloud.dev/mcp  ]8;id=636961;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py\kong_mcp_tool_client.py]8;;\:]8;id=48225;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py#63\63]8;;\

Endpoint : https://kong-948ef205bdausa0ym.kongcloud.dev/mcp
Headers  : X-Kong-API-Key=***, Accept=text/event-stream

list_tools() (static, no network):
  - address_risk_check
  - company_profile
  - control_signal_check
  - entity_lookup
  - evaluate_stop_conditions
  - expand_ownership
  - find_traces_by_entity
  - industry_context_check
  - list_recent_traces
  - ownership_complexity_check
  - resolve_entity
  - retrieve_trace
  - shared_address_check
  - sic_context
  - validate_plan


---

## Part 5 — Factory wiring

`src/app/factory.py` was updated to accept `mcp_mode: str = "local"` instead of
`use_remote_mcp: bool = False`.  The three branches:

```python
if mcp_mode == "kong":
    # Reads KongMCPGatewaySettings; raises EnvironmentError if enabled=False
    mcp_client = KongMCPToolClient(kong_mcp_settings)

elif mcp_mode == "remote":
    mcp_client = RemoteMCPToolClient(get_remote_mcp_url())

else:  # "local" (default)
    mcp_client = MCPToolClient()
```

The chosen `mcp_client` is then wired into all three agents and the orchestrator
in a single place — no per-agent changes.

In [6]:
import inspect
from src.app.factory import create_app_components

# @st.cache_resource wraps the function; unwrap with __wrapped__ if present
_fn = getattr(create_app_components, "__wrapped__", create_app_components)
sig = inspect.signature(_fn)
print("create_app_components signature:")
print(f"  {sig}")
print()
params = sig.parameters
p = params.get("mcp_mode")
if p:
    print(f"  mcp_mode default : {p.default!r}")
    print(f"  mcp_mode kind    : {p.kind.name}")
else:
    print("  mcp_mode param not found — check factory.py")

create_app_components signature:
  (mcp_mode: 'str' = 'local') -> 'AppComponents'

  mcp_mode default : 'local'
  mcp_mode kind    : POSITIONAL_OR_KEYWORD


---

## Part 6 — Live tests

These cells make real HTTP calls through Kong.  They are gated by
`ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true`.

### Test A — `list_tools` via `KongMCPToolClient`

In [7]:
if not live:
    print("SKIPPED — set ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true to run.")
else:
    assert mcp_gw.enabled, "KONG_MCP_GATEWAY_ENABLED must be true"
    assert mcp_gw.proxy_url, "KONG_PROXY_URL must be set"
    assert mcp_gw.api_key, "KONG_MCP_GATEWAY_API_KEY must be set"

    kong_client = KongMCPToolClient(mcp_gw)
    tools = kong_client.list_tools()
    print(f"list_tools() returned {len(tools)} tools (static):")
    for t in tools:
        print(f"  - {t}")
    assert len(tools) == 15, f"Expected 15 tools, got {len(tools)}"
    print()
    print("✓ list_tools OK")

[04/05/26 21:14:38] INFO     Using Kong MCP Gateway                                      ]8;id=738570;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py\kong_mcp_tool_client.py]8;;\:]8;id=832240;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py#62\62]8;;\

                    INFO     Endpoint: https://kong-948ef205bdausa0ym.kongcloud.dev/mcp  ]8;id=810540;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py\kong_mcp_tool_client.py]8;;\:]8;id=383188;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py#63\63]8;;\

list_tools() returned 15 tools (static):
  - address_risk_check
  - company_profile
  - control_signal_check
  - entity_lookup
  - evaluate_stop_conditions
  - expand_ownership
  - find_traces_by_entity
  - industry_context_check
  - list_recent_traces
  - ownership_complexity_check
  - resolve_entity
  - retrieve_trace
  - shared_address_check
  - sic_context
  - validate_plan

✓ list_tools OK


### Test B — `call_tool` (entity_lookup) via `KongMCPToolClient`

This test calls `entity_lookup` through the full Kong → Railway → Neo4j path.
The Railway MCP server must be running and the Neo4j database must be accessible.

In [8]:
if not live:
    print("SKIPPED — set ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true to run.")
else:
    assert mcp_gw.enabled, "KONG_MCP_GATEWAY_ENABLED must be true"
    assert mcp_gw.proxy_url, "KONG_PROXY_URL must be set"
    assert mcp_gw.api_key, "KONG_MCP_GATEWAY_API_KEY must be set"

    kong_client = KongMCPToolClient(mcp_gw)

    print(f"Calling entity_lookup via Kong: {kong_client._endpoint}")
    result = kong_client.call_tool("entity_lookup", {"name": "ACME"})

    print(f"  tool_name  : {result.tool_name}")
    print(f"  success    : {result.success}")
    print(f"  duration_ms: {result.duration_ms}")
    print(f"  summary    : {result.summary[:200]}")
    if result.data:
        print(f"  data keys  : {list(result.data.keys()) if isinstance(result.data, dict) else type(result.data).__name__}")
    print()
    assert result.tool_name == "entity_lookup", "Unexpected tool_name in result"
    print("✓ call_tool via Kong returned a ToolResult")

[04/05/26 21:14:40] INFO     Using Kong MCP Gateway                                      ]8;id=767298;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py\kong_mcp_tool_client.py]8;;\:]8;id=356089;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py#62\62]8;;\

                    INFO     Endpoint: https://kong-948ef205bdausa0ym.kongcloud.dev/mcp  ]8;id=457467;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py\kong_mcp_tool_client.py]8;;\:]8;id=376887;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py#63\63]8;;\

Calling entity_lookup via Kong: https://kong-948ef205bdausa0ym.kongcloud.dev/mcp


                    INFO     HTTP Request: POST https://kong-948ef205bdausa0ym.kongcloud.dev/mcp    ]8;id=635717;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=605467;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 "                                                                       

                    INFO     Received session ID: 1b6551b12abb402a9a5e838ea388b556           ]8;id=650900;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=907621;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/mcp/client/streamable_http.py#134\134]8;;\

...

[04/05/26 21:14:41] INFO     HTTP Request: POST https://kong-948ef205bdausa0ym.kongcloud.dev/mcp    ]8;id=737390;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=992878;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 "                                                                       

                    INFO     HTTP Request: POST https://kong-948ef205bdausa0ym.kongcloud.dev/mcp    ]8;id=89131;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=722936;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 "                                                                       

                    INFO     HTTP Request: DELETE https://kong-948ef205bdausa0ym.kongcloud.dev/mcp  ]8;id=226510;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=490409;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 404 Not Found"                                                              

                    WARNING  Session termination failed: 404                                 ]8;id=183200;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=904669;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/mcp/client/streamable_http.py#435\435]8;;\

  tool_name  : entity_lookup
  success    : True
  duration_ms: 199.5
  summary    : Found 10 company match(es) for 'ACME'. Top match: 'ACME X.S.LTD.' (number: 02101879, status: Active).
  data keys  : list

✓ call_tool via Kong returned a ToolResult


### Test C — `company_profile` via `KongMCPToolClient`

A second tool call to confirm routing is consistent across tool types.

In [9]:
if not live:
    print("SKIPPED — set ENABLE_LIVE_KONG_NOTEBOOK_TESTS=true to run.")
else:
    assert mcp_gw.enabled, "KONG_MCP_GATEWAY_ENABLED must be true"

    kong_client = KongMCPToolClient(mcp_gw)

    print(f"Calling company_profile via Kong: {kong_client._endpoint}")
    result = kong_client.call_tool("company_profile", {"company_name": "ACME"})

    print(f"  tool_name  : {result.tool_name}")
    print(f"  success    : {result.success}")
    print(f"  duration_ms: {result.duration_ms}")
    print(f"  summary    : {result.summary[:200]}")
    print()
    print("✓ company_profile via Kong OK")

[04/05/26 21:14:45] INFO     Using Kong MCP Gateway                                      ]8;id=515393;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py\kong_mcp_tool_client.py]8;;\:]8;id=722169;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py#62\62]8;;\

                    INFO     Endpoint: https://kong-948ef205bdausa0ym.kongcloud.dev/mcp  ]8;id=159119;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py\kong_mcp_tool_client.py]8;;\:]8;id=761293;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py#63\63]8;;\

Calling company_profile via Kong: https://kong-948ef205bdausa0ym.kongcloud.dev/mcp


                    INFO     HTTP Request: POST https://kong-948ef205bdausa0ym.kongcloud.dev/mcp    ]8;id=57997;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=635851;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 "                                                                       

                    INFO     Received session ID: af63846234784a1cbdbdc8492553a13b           ]8;id=374799;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=231746;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/mcp/client/streamable_http.py#134\134]8;;\

...

                    INFO     HTTP Request: POST https://kong-948ef205bdausa0ym.kongcloud.dev/mcp    ]8;id=872700;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=384139;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 "                                                                       

                    INFO     HTTP Request: POST https://kong-948ef205bdausa0ym.kongcloud.dev/mcp    ]8;id=289175;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=830073;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 "                                                                       

                    INFO     HTTP Request: DELETE https://kong-948ef205bdausa0ym.kongcloud.dev/mcp  ]8;id=174464;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=352348;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 404 Not Found"                                                              

                    WARNING  Session termination failed: 404                                 ]8;id=828776;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=688415;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/mcp/client/streamable_http.py#435\435]8;;\

  tool_name  : company_profile
  success    : True
  duration_ms: 96.8
  summary    : No company found with exact name 'ACME'.

✓ company_profile via Kong OK


---

## Part 7 — Fail-closed behavior

`KongMCPToolClient` does **not** return a failed `ToolResult` on error.
It raises `RuntimeError` with a structured message so there is no silent
fallback to direct remote MCP.

The error message always includes:
- The tool that was called
- The Kong endpoint used
- The reason (HTTP status, connection error, etc.)
- A suggestion to disable the flag if Kong is unhealthy

This cell demonstrates the behavior with a deliberately broken endpoint.

In [10]:
from dataclasses import replace as _replace
from src.config import KongMCPGatewaySettings

# Construct settings pointing at an unreachable endpoint
bad_settings = KongMCPGatewaySettings(
    enabled=True,
    proxy_url="https://this-does-not-exist.example.com",
    route_path="/mcp",
    api_key="test-key",
    upstream_url="",
)

bad_client = KongMCPToolClient(bad_settings)

print("Demonstrating fail-closed behavior with a bad endpoint...")
try:
    bad_client.call_tool("entity_lookup", {"name": "ACME"})
    print("ERROR: expected RuntimeError but call succeeded")
except RuntimeError as exc:
    print(f"✓ RuntimeError raised as expected:")
    print()
    for line in str(exc).splitlines():
        print(f"  {line}")
except Exception as exc:
    print(f"Unexpected exception type {type(exc).__name__}: {exc}")

[04/05/26 21:14:52] INFO     Using Kong MCP Gateway                                      ]8;id=225854;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py\kong_mcp_tool_client.py]8;;\:]8;id=728472;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py#62\62]8;;\

                    INFO     Endpoint: https://this-does-not-exist.example.com/mcp       ]8;id=402117;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py\kong_mcp_tool_client.py]8;;\:]8;id=124641;file:///Users/emilpastor/Documents/github/entity-risk-ai/notebooks/../src/clients/kong_mcp_tool_client.py#63\63]8;;\

Demonstrating fail-closed behavior with a bad endpoint...
✓ RuntimeError raised as expected:

  Kong MCP Gateway call failed for tool 'entity_lookup'.
  Endpoint: https://this-does-not-exist.example.com/mcp
  Reason: unhandled errors in a TaskGroup (1 sub-exception)
  Suggestion: check Kong gateway health or set KONG_MCP_GATEWAY_ENABLED=false to disable.


---

## Part 8 — Regression: local and remote MCP still work

Phase 508 does not alter `MCPToolClient` or `RemoteMCPToolClient`.
The only change to `factory.py` is the signature change from
`use_remote_mcp: bool` to `mcp_mode: str` — backward-compatible at
the app level since the call site in `layout.py` was updated in the
same commit.

This cell verifies local MCP still initializes correctly.

In [11]:
from src.clients.mcp_tool_client import MCPToolClient

local_client = MCPToolClient()
tools = local_client.list_tools()
print(f"Local MCPToolClient: {len(tools)} tools available")
assert len(tools) == 15, f"Expected 15, got {len(tools)}"
print("✓ Local MCP unaffected")

# RemoteMCPToolClient: instantiation is always safe (no connection at init time)
from src.clients.remote_mcp_tool_client import RemoteMCPToolClient
remote_client = RemoteMCPToolClient("https://example.com/mcp")
tools = remote_client.list_tools()
print(f"RemoteMCPToolClient : {len(tools)} tools available (static)")
print("✓ Remote MCP client interface unaffected")

Local MCPToolClient: 15 tools available
✓ Local MCP unaffected
RemoteMCPToolClient : 15 tools available (static)
✓ Remote MCP client interface unaffected


---

## Part 9 — UI sidebar: Kong as a third option

The sidebar in `src/app/layout.py` now builds the options list dynamically:

```python
_mcp_options = ["local", "remote"]
if kong_mcp_enabled:          # KONG_MCP_GATEWAY_ENABLED=true
    _mcp_options.append("kong")
```

| `KONG_MCP_GATEWAY_ENABLED` | Sidebar options |
|---|---|
| `false` (default) | Local (in-process), Remote MCP Server |
| `true` | Local (in-process), Remote MCP Server, **Kong MCP Gateway** |

The option only appears when explicitly enabled — no accidental exposure.

When selected, `layout.py` calls:

```python
components = create_app_components(mcp_mode="kong")
```

This is cached by `@st.cache_resource` on the `mcp_mode` argument, so
switching between local/remote/kong creates three isolated component graphs.

In [12]:
# Confirm the env flag determines option visibility
kong_enabled = os.getenv("KONG_MCP_GATEWAY_ENABLED", "").strip().lower() == "true"

options = ["local", "remote"]
if kong_enabled:
    options.append("kong")

print("Sidebar MCP options given current .env:")
for opt in options:
    label = {
        "local": "Local (in-process)",
        "remote": "Remote MCP Server",
        "kong": "Kong MCP Gateway",
    }[opt]
    print(f"  - {label}  [{opt}]")

if not kong_enabled:
    print()
    print("Kong option is hidden (KONG_MCP_GATEWAY_ENABLED is not true).")

Sidebar MCP options given current .env:
  - Local (in-process)  [local]
  - Remote MCP Server  [remote]
  - Kong MCP Gateway  [kong]


---

## Part 10 — Rollback

Phase 508 rollback is non-destructive.  All three paths remain independently
operational — disabling Kong just removes the third UI option.

### Option A — disable the env flag (minimal)

```dotenv
KONG_MCP_GATEWAY_ENABLED=false
```

Effect:
- Kong option disappears from the sidebar
- `create_app_components(mcp_mode="kong")` raises `EnvironmentError` if called
- Local and remote MCP paths are completely unaffected
- No code changes required

### Option B — revert the three changed files

Files touched in Phase 508:

```
src/clients/kong_mcp_tool_client.py   ← new file (delete)
src/app/factory.py                    ← mcp_mode param + kong branch
src/app/layout.py                     ← sidebar kong option
scripts/smoke_kong_mcp.py             ← new file (delete)
```

Reverting these four files fully restores Phase 507 behaviour.

In [ ]:
# Confirm rollback path: flag off → factory raises, not silently falls back
from src.config import KongMCPGatewaySettings

disabled_settings = KongMCPGatewaySettings(
    enabled=False,
    proxy_url="https://example.com",
    route_path="/mcp",
    api_key="some-key",
    upstream_url="",
)

# Simulate what factory.py does when mcp_mode="kong" but flag is false
if not disabled_settings.enabled:
    print("✓ Factory would raise EnvironmentError when mcp_mode='kong' but flag is false.")
    print("  This is the expected fail-closed behavior at wiring time.")
else:
    print("Flag is true — normal operation.")